<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/08_Uplift_Modeling_Causal_Inference/01_Discount_Optimization_TLearner_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 8: Causal Inference & Uplift Modeling

## 1. The Real Business Problem: The Margin Trap
In Phase 7, we mathematically identified that our "At-Risk Loyalists" belong entirely to the Broad Middle segment. The standard retail reflex is to send a blanket promotional discount (e.g., "$10 off your next $50 basket") to this entire group to win them back.

However, grocery profit margins are notoriously razor-thin (typically 1% to 3%). When a business sends a promotion, every recipient falls into one of four invisible behavioral quadrants:
* **The Persuadables:** Will only return if they receive the coupon. *(High ROI).*
* **The Sure Things:** Were going to return regardless of the coupon. *(Massive margin cannibalization if we send it).*
* **The Lost Causes:** Have permanently switched stores. *(Wasted marketing spend).*
* **The Sleeping Dogs:** Inactive customers who get annoyed by marketing spam and actively unsubscribe. *(Negative ROI).*

Standard machine learning cannot solve this because it only predicts an *outcome* (Will they buy?). To protect profit margins, we must predict *causality* (Would they have bought *without* the coupon?). We need to isolate **The Persuadables**.

## 2. The Solution: T-Learner Architecture
We do not need to run expensive, live randomized control trials. We can synthesize historical campaign data (customers who received discounts vs. those who did not) using a **Two-Learner (T-Learner)** approach.

Instead of building one monolithic neural network, we train two distinct, lightweight regression models to predict 30-day future spend:
* **Model $C$ (Control):** Trained exclusively on historical customers who received NO promotional campaigns. Learns baseline spending behavior.
* **Model $T$ (Treatment):** Trained exclusively on historical customers who DID receive promotional campaigns. Learns incentivized spending behavior.

## 3. The Causal Subtraction & Actionable Output
To evaluate our current At-Risk customers, we pass their data through *both* models simultaneously to calculate the **Individual Treatment Effect (ITE)**:

$$Uplift(x) = \hat{Y}_T(x) - \hat{Y}_C(x)$$

* If Model T predicts $60 spend and Model C predicts $10 spend, the Uplift is **+$50**. (A Persuadable).
* If Model T predicts $80 spend and Model C predicts $80 spend, the Uplift is **$0**. (A Sure Thing).

**The Final Business Rule:** We eliminate margin cannibalization by establishing a strict financial threshold. We will only issue the $10 discount to customers where their predicted $Uplift$ strictly exceeds $10.

## Step 1: Engineering the Causal Dataset (Timeline Split)

To predict causality, we must prevent data leakage by strictly separating our observation period from our target measurement period.

* **The Cutoff:** We separate the 1-year dataset by reserving the final 90 days as the Target Window.
* **The Features (X):** Using the pre-cutoff data, we calculate each household's historical spend and trip frequency.
* **The Treatment (T):** We query the `campaigns` table. If a household received marketing prior to the cutoff, they are assigned to the Treatment group (`T = 1`). Otherwise, they belong to the Control group (`T = 0`).
* **The Target (Y):** We calculate the total revenue generated by each household strictly within the final 90-day Target Window.

In [13]:
!pip install completejourney_py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 52.8 MB/s eta 0:00:00


In [14]:
import pandas as pd
from completejourney_py import get_data

print("Fetching raw tables...")
data = get_data()
transactions = data['transactions']
campaigns = data['campaigns']

# 1. Enforce date formatting and define the strict chronological cutoff
transactions['transaction_timestamp'] = pd.to_datetime(transactions['transaction_timestamp'])
max_date = transactions['transaction_timestamp'].max()
cutoff_date = max_date - pd.Timedelta(days=90)

print(f"Dataset split! Observation ends on {cutoff_date.date()}. Target Window is the final 90 days.")

# 2. Extract The Observation Window (Pre-Cutoff)
pre_cutoff_data = transactions[transactions['transaction_timestamp'] < cutoff_date]

# Build baseline customer features (X)
causal_df = pre_cutoff_data.groupby('household_id').agg(
    Historical_Spend=('sales_value', 'sum'),
    Historical_Trips=('basket_id', 'nunique')
).reset_index()

# 3. Define the Treatment Vector (T)
# Identify households that received a campaign during the observation window
treated_households = campaigns['household_id'].unique()
causal_df['Treatment_Flag'] = causal_df['household_id'].isin(treated_households).astype(int)

# 4. Extract The Target Window (Post-Cutoff)
post_cutoff_data = transactions[transactions['transaction_timestamp'] >= cutoff_date]

# Calculate the exact revenue generated in the target window (Y)
future_revenue = post_cutoff_data.groupby('household_id').agg(
    Target_90_Day_Spend=('sales_value', 'sum')
).reset_index()

# 5. Merge into the final analytical dataframe
causal_df = causal_df.merge(future_revenue, on='household_id', how='left')

# If a customer didn't shop in the final 90 days, their future spend is $0
causal_df['Target_90_Day_Spend'] = causal_df['Target_90_Day_Spend'].fillna(0)

print("\n✅ Causal Dataset Engineered:")
display(causal_df.head(10))
display(causal_df['Treatment_Flag'].value_counts().rename(index={0: "Control (0)", 1: "Treatment (1)"}))

Fetching raw tables...
Dataset split! Observation ends on 2017-10-03. Target Window is the final 90 days.

✅ Causal Dataset Engineered:


,household_id,Historical_Spend,Historical_Trips,Treatment_Flag,Target_90_Day_Spend
0,1,1796.67,38,1,618.89
1,2,693.43,13,1,330.69
2,3,951.13,17,1,75.50
3,4,300.21,14,1,141.93
4,5,236.69,16,0,62.98
5,6,2764.07,119,1,700.80
6,7,1506.70,21,1,445.67
7,8,2325.32,49,1,755.49
8,9,418.51,7,0,202.22
9,10,29.96,1,0,0.00


,count
Treatment_Flag,
Treatment (1),1558
Control (0),889
